<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">00 - Course Overview</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Databricks Performance Optimization
#### Duration: 5 minutes

## Overview

Diagnose and fix Databricks performance problems that waste money. Platform-first: let the defaults work, verify in the Query Profile, tune only what remains.

- **Measurement**: Query Profiles, Performance Insights, `system.query.history`, honest benchmarking
- **Data layout**: small files, self-tuning managed tables, liquid clustering
- **Workload patterns**: shuffle, skew, spill, row explosions, driver traps, UDF pitfalls
- **Compute selection**: serverless prerequisites, Photon verification
- **Cost attribution**: billing system tables, usage tags, DBU-to-dollar conversion

Every optimization is judged on **latency** and **DBUs per run**.

## Terminal Objectives

By the end of this course, you will be able to:

1. **Measure first**: read a Query Profile, rank bottlenecks with Performance Insights, find worst offenders in system tables and benchmark honestly.
2. **Fix data layout**: diagnose small-file problems, verify self-tuning with `DESCRIBE DETAIL` and apply liquid clustering.
3. **Fix workload patterns**: recognize shuffle, skew, spill, and row explosions in the profile and intervene where AQE and Photon leave a gap.
4. **Fix driver and UDF pitfalls**: spot code that serializes on the driver, pick the right UDF type and verify Photon coverage.
5. **Choose compute**: apply the serverless prerequisites, identify classic-only cases and confirm Photon engaged.
6. **Attribute cost**: convert DBUs to dollars with billing system tables, tag serverless spend and use `total_task_duration_ms` as an immediate proxy.

## A. Prerequisites

Before starting this course, learners should be comfortable with the following:

- **PySpark DataFrames and Spark SQL**: writing transformations, joins, aggregations and actions. This course doesn't teach PySpark or SQL basics.
- **Unity Catalog**: creating and navigating catalogs, schemas and managed tables. The course creates a per-user catalog and schema for demo and lab data.
- **Databricks notebooks**: running cells, attaching compute and using `%run` to call setup scripts.
- **SQL queries**: reading and writing SQL for system table queries, aggregations and joins. The measurement and cost sections rely heavily on SQL.
- **Lakeflow Jobs**: basic familiarity with scheduling and running notebook workloads as jobs.

## B. Workspace Setup Information



<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;">This content assumes you have:</p>

- Access to <strong>Serverless Compute</strong> (all notebooks run on serverless notebook compute, environment version 5).
- <strong>Unity Catalog</strong> enabled in your workspace and ability to create a catalog.
- <strong>System tables</strong> enabled in the workspace (the Workspace-Setup notebook creates filtered views over them). 
</div>
</div>
</div>

### B1. Databricks Provided Vocareum Workspace (Recommended)

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <div style="color:#333;">

- If you are running this notebook in a <strong>Databricks Academy provided Vocareum workspace</strong>, your Unity Catalog catalog is already created for you.

- Your catalog name matches your Vocareum username and looks like: <strong>labuser12345</strong> (series of unique numbers)

- This course does not require <strong>Marketplace</strong> data. All demo and lab data is generated synthetically by the setup scripts.

  </div>
</div>

### B2. Databricks Free Edition or Outside Workspaces (*as is*)

##### Databricks Free Edition or Other Workspaces may work for this, but it is provided **as is** and support is not guaranteed.  

Some features may not be available depending on the capabilities of Databricks Free Edition or your Workspace.

Please read below to setup your environment

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
<div style="color:#333;">

#### Catalog Information

- If you are running this notebook in your own Databricks workspace or Databricks Free Edition, the setup will <strong>create a Unity Catalog catalog and schema for you</strong>. 

- The <strong>Create Catalog</strong> permission is required.

- The catalog name is derived from your Databricks username and follows this pattern: <strong>labuser_username</strong>

#### Access Marketplace Data

Marketplace data is not required.

</div>
</div>

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">

  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Troubleshooting Setup - Your Workspace Can't Create Catalogs
  </strong>
<details>
  <div style="color:#333;">

If you do not have permission to create a new catalog but already have one available, you can explicitly specify an existing catalog by updating the following:

- In this notebook: `Classroom-Setup-Common-Python`.
  - Specify your catalog name in the argument here (last cell): `my_catalog = build_user_catalog(catalog_forced='YOUR_CATALOG_NAME')`

  </div>
</details>
</div>

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
<strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Do Not Run in Production Environments</strong>

<div style="color:#333;">
<ul>
<li>Only run this course in <strong>development or sandbox workspaces</strong>.</li>
<li>Do not run in production environments. The setup scripts create catalogs, schemas and tables in your workspace.</li>
</ul>
</div>
</div>

## C. Course Agenda

<div style="max-width: 1200px; margin: 0 auto; font-family: sans-serif;">
<div style="background: #F9F7F4; border-radius: 10px; padding: 20px 24px; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">

<style>
.agenda-table td, .agenda-table th {
  font-size: 14pt !important;
}
</style>

<table class="agenda-table" style="width: 100%; border-collapse: collapse; line-height: 1.5;">
  <thead>
    <tr style="background: #1B5162; color: white;">
      <th style="padding: 10px 14px; text-align: center; border: 1px solid #EEEDE9; width: 50px;">#</th>
      <th style="padding: 10px 14px; text-align: center; border: 1px solid #EEEDE9; width: 80px;">Type</th>
      <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">Module Name</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background: white;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">1</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #e3f2fd; color: #1976d2; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Lecture</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">The Performance Landscape</td>
    </tr>
    <tr style="background: #F9F7F4;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">2</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #fff3e0; color: #e65100; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Demo</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">Measure First: Query Profiles and Benchmarks</td>
    </tr>
    <tr style="background: white;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">3</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #fff3e0; color: #e65100; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Demo</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">Data Layout: Small Files and Self-Tuning Tables</td>
    </tr>
    <tr style="background: #F9F7F4;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">4</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #fff3e0; color: #e65100; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Demo</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">Liquid Clustering vs Partitioning</td>
    </tr>
    <tr style="background: white;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">5</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #fff3e0; color: #e65100; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Demo</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">Caching and Intermediate Results</td>
    </tr>
    <tr style="background: #F9F7F4;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">6</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #e8f5e9; color: #2e7d32; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Lab</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">Lab 1 - Layout</td>
    </tr>
    <tr style="background: white;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">7</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #fff3e0; color: #e65100; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Demo</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">Shuffle, Skew, Spill and Explosion</td>
    </tr>
    <tr style="background: #F9F7F4;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">8</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #fff3e0; color: #e65100; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Demo</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">Python, Driver and UDF Pitfalls</td>
    </tr>
    <tr style="background: white;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">9</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #e3f2fd; color: #1976d2; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Lecture</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">Serverless First and Photon</td>
    </tr>
    <tr style="background: #F9F7F4;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">10</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #fff3e0; color: #e65100; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Demo</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">Cost & Cost Attribution</td>
    </tr>
    <tr style="background: white;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">11</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #e8f5e9; color: #2e7d32; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Lab</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">Lab 2 - Cost</td>
    </tr>
    <tr style="background: #F9F7F4;">
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center; font-weight: 700; color: #1B5162;">-</td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9; text-align: center;"><span style="background: #f3e5f5; color: #7b1fa2; padding: 2px 8px; border-radius: 4px; font-weight: 600;">Reference</span></td>
      <td style="padding: 8px 14px; border: 1px solid #EEEDE9;">Appendix A - Reading the Query Profile</td>
    </tr>
  </tbody>
</table>

</div>
</div>


### Requirements

Please review the following requirements before starting the lesson:

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <div style="color:#333;">

- Use Databricks Runtime version: **`Serverless`** for running all demo and lab notebooks.

  </div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>